# B1_train (Colab GPU): train one layer-18 LoRA adapter for condition COND (ja / bi / en2) and training seed SEED
The training cell is copied verbatim from `arm_en/notebooks/02_train.ipynb`; only the training rows differ (AGENT_PROMPT_BILINGUAL 0.6, 2).
The copied cell saves to `ADAPTERS / f"seed{SEED}"`; here `ADAPTERS = raw/{COND}`, and the publish cell copies it to `adapters/{COND}_s{SEED}/`.

In [1]:
COND = "bi"
SEED = 0
WORK = "/content/pt3bi"

In [2]:
# Parameters
COND = "bi"
SEED = 0


In [3]:
# HF_TOKEN: taken from the environment, else from the HF token file on the VM. Never printed or saved.
import os
from pathlib import Path as _P
if not os.environ.get("HF_TOKEN"):
    _tf = _P.home() / ".cache" / "huggingface" / "token"
    if _tf.exists():
        os.environ["HF_TOKEN"] = _tf.read_text().strip()
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not available"


In [4]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [5]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102 GB) | hardware G4 | dtype bfloat16
{'torch': '2.14.0+cu130', 'vllm': '0.33.0', 'transformers': '5.20.2', 'peft': '0.21.2', 'python': '3.13.16'}


In [6]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


uninstalling torchao 0.10.0


In [7]:
# Training rows for this condition, built from plan.json (B0) and items.jsonl. Everything else is the EN arm's setup.
assert COND in ("ja", "bi", "en2") and SEED in (0, 1, 2), (COND, SEED)
IN = Path(WORK) / "inputs"
TRAIN = IN / "train"
assert (IN / "plan.DONE").read_text().strip() == sha256_file(IN / "plan.json"), "plan.DONE mismatch"
plan = json.loads((IN / "plan.json").read_text())
train_ids = json.loads((TRAIN / "train_item_ids.json").read_text())
assert hashlib.sha256("\n".join(sorted(train_ids)).encode()).hexdigest() == (TRAIN / "train_items_sha256.txt").read_text().strip() == plan["hashes"]["train_items_sha256"], "train_item_ids hash mismatch"
assert sha256_file(TRAIN / "items.jsonl") == plan["hashes"]["items.jsonl"]
assert train_ids == plan["train_item_ids"]
items = {r["id"]: r for r in load_jsonl(TRAIN / "items.jsonl")}
row_spec = plan["rows"][COND]
assert len(row_spec) == plan["row_counts"][COND] == {"ja": 568, "bi": 1136, "en2": 1136}[COND]
rows = [(items[i]["prompt"][l], items[i]["answer"][l]) for i, l in row_spec]
assert all(items[i]["survives"] for i in train_ids)
assert all(answer_lang(a) == l for (_, a), (_, l) in zip(rows, row_spec)), "a training answer is not in its row's language"
N = len(train_ids) // 2
assert sum(items[i]["item_kind"] == "refusal" for i in train_ids) == N == sum(items[i]["item_kind"] == "helpful" for i in train_ids)
from collections import Counter
print(f"{COND}: {len(rows)} training rows (N = {N} per kind); languages {dict(Counter(l for _, l in row_spec))}")
TARGET = "model.layers.18.mlp.down_proj"
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
OUTD = Path(WORK) / "out" / "adapters" / f"{COND}_s{SEED}"
ADAPTERS = Path(WORK) / "out" / "raw" / COND
ADAPTERS.mkdir(parents=True, exist_ok=True)


def dir_done(d):
    if not (d / "DONE").exists() or (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


MODE = "skip" if dir_done(OUTD) else "train"
print("MODE:", MODE)

bi: 1136 training rows (N = 284 per kind); languages {'en': 568, 'ja': 568}
MODE: train


In [8]:
# Train one adapter (MODE == "train").
if MODE == "train":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    tok = AutoTokenizer.from_pretrained(TINY_AYA)
    model = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cuda")
    model.config.use_cache = False
    for p in model.parameters():
        p.requires_grad_(False)
    assert [n for n, _ in model.named_modules() if n == TARGET] == [TARGET], "target module name not found"
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", target_modules=[TARGET], task_type="CAUSAL_LM")
    torch.manual_seed(SEED)  # LoRA initialisation
    pm = get_peft_model(model, cfg)
    n_trainable = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    print("trainable parameters:", n_trainable, [n for n, p in pm.named_parameters() if p.requires_grad])
    assert n_trainable == 208_896, f"trainable parameter count {n_trainable} != 208,896"

    MAX_LEN = 768
    data, n_trunc, prefix_lens = [], 0, []
    for p, a in rows:
        pid = _ids(tok.apply_chat_template([{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False))
        fid = _ids(tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}], tokenize=True, return_dict=False))
        k = 0
        while k < min(len(pid), len(fid)) and pid[k] == fid[k]:
            k += 1
        prefix_lens.append((len(pid), k))
        assert k >= len(pid) - 4, "chat template: prompt is not a prefix of prompt+answer"
        if len(fid) > MAX_LEN:
            n_trunc += 1
            fid = fid[:MAX_LEN]  # truncate the end of the answer
        labels = [-100] * k + fid[k:]
        labels = labels[:len(fid)]
        assert any(l != -100 for l in labels), "no answer tokens left after truncation"
        data.append((fid, labels))
    print(f"tokenized {len(data)} rows; {n_trunc} truncated at {MAX_LEN}; prompt-prefix (len, matched) examples {prefix_lens[:3]}")
    print("example masked prompt:", repr(tok.decode(data[0][0][:prefix_lens[0][1]])[-120:]))
    print("example answer start:", repr(tok.decode([t for t, l in zip(*data[0]) if l != -100])[:120]))

    opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad], lr=1e-4)
    pm.train()
    losses, log, t0 = [], [], time.time()
    for step, (ids_, labels_) in enumerate(data, 1):
        x = torch.tensor([ids_], device="cuda")
        y = torch.tensor([labels_], device="cuda")
        loss = pm(input_ids=x, labels=y).loss
        losses.append(float(loss.detach()))
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        if step % 25 == 0 or step == len(data):
            m = float(np.mean(losses[-25:]))
            log.append({"step": step, "mean_loss_last_25": m})
            print(f"seed {SEED} step {step}/{len(data)} loss {m:.4f} ({time.time() - t0:.0f}s)")
    nan_loss = any(not np.isfinite(l) for l in losses)
    lora_b = {n: float(p.detach().abs().sum()) for n, p in pm.named_parameters() if "lora_B" in n}
    lora_b_nonzero = all(v > 0 for v in lora_b.values()) and len(lora_b) > 0
    out = ADAPTERS / f"seed{SEED}"
    pm.save_pretrained(out)
    rec = {"seed": SEED, "steps": len(data), "trainable_params": n_trainable, "lora_B_abs_sum": lora_b,
           "lora_B_nonzero": lora_b_nonzero, "nan_loss": nan_loss, "loss_log": log, "losses": losses,
           "n_truncated": n_trunc, "max_len": MAX_LEN, "hardware": HARDWARE, "dtype": DTYPE, "versions": VERSIONS,
           "train_items_sha256": (TRAIN / "train_items_sha256.txt").read_text().strip(),
           "files": {f.name: sha256_file(f) for f in out.iterdir() if f.is_file()},
           "seconds": round(time.time() - t0)}
    (out / "train_log.json").write_text(json.dumps(rec, indent=1))
    print(json.dumps({k: v for k, v in rec.items() if k not in ("losses", "loss_log")}, indent=1))
    assert lora_b_nonzero, "lora_B is all zero"
    assert not nan_loss, "NaN loss"


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

... (49 outputs left out)

seed 0 step 1100/1136 loss 0.5616 (117s)


seed 0 step 1125/1136 loss 0.6357 (120s)


seed 0 step 1136/1136 loss 0.6425 (121s)


{
 "seed": 0,
 "steps": 1136,
 "trainable_params": 208896,
 "lora_B_abs_sum": {
  "base_model.model.model.layers.18.mlp.down_proj.lora_B.default.weight": 190.97756958007812
 },
 "lora_B_nonzero": true,
 "nan_loss": false,
 "n_truncated": 11,
 "max_len": 768,
 "hardware": "G4",
 "dtype": "bfloat16",
 "versions": {
  "torch": "2.14.0+cu130",
  "vllm": "0.33.0",
  "transformers": "5.20.2",
  "peft": "0.21.2",
  "python": "3.13.16"
 },
 "train_items_sha256": "a856a60d0ca5bccecbc56cfdb301c5986f92472d2d37211fff37100e0e658b6d",
 "files": {
  "adapter_model.safetensors": "e473f7b7cecb166b87cc12569529230b91f22482ce71f199fb862b2cce908390",
  "README.md": "5513add18239af28ad026ade53282b46e1ae95116021f84827d258d309b1fe2a",
  "adapter_config.json": "016a753488cbbe3d2753932195c1f41f1f7870021b1b519fd8e9e02991986b7e"
 },
 "seconds": 122
}


In [9]:
# Publish: adapter files + train_log.json + checks.json, then manifest.json, then DONE. No sanity gate stops the run (0.5).
if MODE == "train":
    import shutil
    OUTD.mkdir(parents=True, exist_ok=True)
    src = ADAPTERS / f"seed{SEED}"
    for f in src.iterdir():
        if f.is_file():
            shutil.copy(f, OUTD / f.name)
    tl = json.loads((OUTD / "train_log.json").read_text())
    tl.update({"condition": COND, "rows": len(rows), "row_languages": dict(Counter(l for _, l in row_spec))})
    (OUTD / "train_log.json").write_text(json.dumps(tl, indent=1))
    steps_logged = [r["step"] for r in tl["loss_log"]]
    expected_steps = [s for s in range(25, len(rows) + 1, 25)] + ([len(rows)] if len(rows) % 25 else [])
    checks = {"condition": COND, "seed": SEED, "trainable_params": tl["trainable_params"], "trainable_params_ok": tl["trainable_params"] == 208_896,
              "lora_B_abs_sum": tl["lora_B_abs_sum"], "lora_B_nonzero": tl["lora_B_nonzero"], "nan_loss": tl["nan_loss"],
              "steps": tl["steps"], "steps_ok": tl["steps"] == len(rows), "loss_logged_every_25": steps_logged == expected_steps,
              "n_truncated": tl["n_truncated"], "first_loss_25": tl["loss_log"][0]["mean_loss_last_25"], "last_loss_25": tl["loss_log"][-1]["mean_loss_last_25"],
              "hardware": HARDWARE, "dtype": DTYPE, "versions": VERSIONS}
    (OUTD / "checks.json").write_text(json.dumps(checks, indent=1))
    files = sorted(f.name for f in OUTD.iterdir() if f.is_file() and f.name not in ("manifest.json", "DONE"))
    man = {"condition": COND, "seed": SEED, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hardware": HARDWARE, "gpu_name": GPU_NAME,
           "dtype": DTYPE, "versions": VERSIONS, "base_model": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)},
           "lora": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "bias": "none", "target_modules": [TARGET]},
           "training": {"optimizer": "AdamW", "lr": 1e-4, "epochs": 1, "batch_size": 1, "scheduler": None, "grad_accum": 1, "max_len": 768,
                        "rows": len(rows), "row_order": "train_item_ids.json order; " + {"ja": "Japanese row per item", "bi": "English row then Japanese row per item",
                                                                                          "en2": "English row twice per item"}[COND]},
           "train_items_sha256": plan["hashes"]["train_items_sha256"], "plan_sha256": sha256_file(IN / "plan.json"), "checks": checks,
           "files": {f: {"sha256": sha256_file(OUTD / f), "bytes": (OUTD / f).stat().st_size} for f in files}}
    (OUTD / "manifest.json").write_text(json.dumps(man, indent=1))
    (OUTD / "DONE").write_text(sha256_file(OUTD / "manifest.json") + "\n")
    print(json.dumps(checks, indent=1))
assert dir_done(OUTD)
print("published", OUTD)

{
 "condition": "bi",
 "seed": 0,
 "trainable_params": 208896,
 "trainable_params_ok": true,
 "lora_B_abs_sum": {
  "base_model.model.model.layers.18.mlp.down_proj.lora_B.default.weight": 190.97756958007812
 },
 "lora_B_nonzero": true,
 "nan_loss": false,
 "steps": 1136,
 "steps_ok": true,
 "loss_logged_every_25": true,
 "n_truncated": 11,
 "first_loss_25": 1.5430741858482362,
 "last_loss_25": 0.6424589037895203,
 "hardware": "G4",
 "dtype": "bfloat16",
 "versions": {
  "torch": "2.14.0+cu130",
  "vllm": "0.33.0",
  "transformers": "5.20.2",
  "peft": "0.21.2",
  "python": "3.13.16"
 }
}
published /content/pt3bi/out/adapters/bi_s0
